In [69]:
import json
import sys
sys.path.append(r"D:\Desktop_fake\MawileBot\home\MawileBot\src")
from poke_lib import get_poke_bst, similar_pokemon_name, next_gym, generate_all_types_combo, poke_cell_gym
from BeatlesBoy_utils import find_evo_at_level_x,load_x_from_json, calculate_bonus_via_types
import random

## Da lanciare in D:\Desktop_fake\MawileBot (NON D:\Desktop_fake\MawileBot\home\MawileBot )

In [70]:
async def pokemon_utility_old(pokemon,lvl):
    if pokemon is  None:
        return 0
    #TODO: implement a better utility function
        
    pokemon = await similar_pokemon_name(pokemon.lower(), r = False)
    fully_evo = await find_evo_at_level_x(pokemon, 100)
    max_bst = await get_poke_bst(fully_evo)
    if fully_evo in (await load_x_from_json("mega")):
        max_bst = await get_poke_bst(fully_evo+'-mega')
        print(f"BST di Mega {await find_evo_at_level_x(pokemon, 100)} a livello 100: {max_bst}")

    else:
        print(f"BST di {await find_evo_at_level_x(pokemon, 100)} a livello 100: {max_bst}")

    utility_bst = max_bst/620*10
    utility_lvl = lvl/10

    utility = utility_bst * 0.7 + utility_lvl * 0.3
    
    return round(utility, 2)  # 0-10 scale


In [ ]:
async def pokemon_utility(pokemon,lvl, catch = False):
    if pokemon is  None:
        return 0
    #TODO: implement a better utility function
        
    pokemon = await similar_pokemon_name(pokemon.lower(), r = False)
    fully_evo = await find_evo_at_level_x(pokemon, 100)
    max_bst = await get_poke_bst(fully_evo)
    if fully_evo in (await load_x_from_json("mega")):
        max_bst = await get_poke_bst(fully_evo+'-mega')
        print(f"BST di Mega {await find_evo_at_level_x(pokemon, 100)} a livello 100: {max_bst}")

    else:
        print(f"BST di {await find_evo_at_level_x(pokemon, 100)} a livello 100: {max_bst}")


    utility_bst = max_bst/620*10
    utility_lvl = lvl/10

    utility = utility_bst * 0.7 + utility_lvl * 0.3

    try:
        utility += await next_gym_bonus(pokemon, lvl, catch)
    except Exception as e:
        print(f"Error calculating next gym bonus: {e}")
        
    return round(utility, 2)  # 0-10 scale (except bonus next gym)

In [72]:
import pypokedex as poke

gym_type = 'rock'
multiplier = 20
low_power = 30
power  = 20
pokemon = 'pikachu'

def win_perc_over_gym(gym_type, low_power, pokemon, power, multiplier):
    all_types_combo = generate_all_types_combo(gym_type)
    wins = 0
    for t in all_types_combo:
        types2 = poke.get(name=pokemon).types
        bonus = calculate_bonus_via_types(t, types2 ,multiplier)
        if power - bonus[0] + bonus[1] > low_power:
            wins += 1
    return wins / len(all_types_combo)

async def next_gym_bonus(pokemon, lvl, catch = False):
    gym_type,multiplier,casella_gym = (await next_gym())
    power = await get_poke_bst(pokemon)*lvl/100
    _, _, enemy_powers, multiplier, _ = poke_cell_gym(casella_gym)
    low_power = min(enemy_powers)
    win_perc = win_perc_over_gym(gym_type, low_power, pokemon, power, multiplier)
    print(f"Win percentage against next gym: {win_perc*100:.2f}%")

    lvl += 5
    gym_type,multiplier,casella_gym = (await next_gym())
    power = await get_poke_bst(pokemon)*lvl/100
    _, _, enemy_powers, multiplier, _ = poke_cell_gym(casella_gym)
    low_power = min(enemy_powers)
    win_perc_plus5 = win_perc_over_gym(gym_type, low_power, pokemon, power, multiplier)    
    print(f"Win percentage against next gym: {win_perc_plus5*100:.2f}%")

    if win_perc_plus5 > 0.75:
        if win_perc < 0.75:
            return 3
        
    if win_perc > 0.75 and catch == True:
        return 4
    
    return 0




In [80]:
print(await pokemon_utility('Pikachu',7))
print(await pokemon_utility('Pikachu',7, catch=True))
print('\n')
print(await pokemon_utility('Decidueye-hisui',16))


Loading mega from: ./home/MawileBot/src\BeatlesBoy_info.json
BST di raichu-alola a livello 100: 485
Win percentage against next gym: 77.78%
Win percentage against next gym: 94.44%
5.69
Loading mega from: ./home/MawileBot/src\BeatlesBoy_info.json
BST di raichu-alola a livello 100: 485
Win percentage against next gym: 77.78%
Win percentage against next gym: 94.44%
9.69


Loading mega from: ./home/MawileBot/src\BeatlesBoy_info.json
BST di decidueye-hisui a livello 100: 530
Win percentage against next gym: 22.22%
Win percentage against next gym: 77.78%
9.46
